77689984D Álvaro García Zafra
78266424T Fernando Castilla Ropero
78114273V Jorge Lopez Molina


## Ejercicio 1

## a.1 
**Idee un medio de generar una clave “aleatoria” de Vigenère tan larga como el mensaje que desee cifrar.**

Para este apartado hemos decidido aportar dos funciones distintas que generan la clave aleatoria de tamaño igual al mensaje a cifrar.

Primeramente se propone un generaClaveVigenereConNoAlfabeticos que genera una clave de Vigenère que incluye caracteres no alfabéticos en las mismas posiciones que el mensaje original. De este modo, la clave generada tiene la misma longitud y estructura que el mensaje original, facilitando su uso en el cifrado y descifrado. Entendemos que esto podría suponer una vulnerabilidad en términos de seguridad, ya que si el atacante conoce la estructura del mensaje, podría deducir partes de la clave. Sin embargo, este método puede ser útil en situaciones donde se desea mantener la estructura del formato del mensaje original.

Por ello, hemos decidido aportar tambien un segundo método, generaClaveVigenereDefault, que genera una clave de Vigenère compuesta únicamente por caracteres alfabéticos. Esta clave tiene una longitud igual al número de caracteres alfabéticos en el mensaje original. Este enfoque mejora la seguridad del cifrado, ya que la clave no revela información sobre la estructura del mensaje original.

Ambas funciones son perfectamente válidas para probarlas con los métodos de cifrado y descifrado que desarrollamos en la tarea del 20% como se verá más adelante.

In [62]:
import random

def generaClaveVigenereConNoAlfabeticos(mensaje):
    clave = ""

    for m in mensaje:
        if not m.isalpha():
            clave += m
        else:
            clave += random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

    return clave

def generaClaveVigenereDefault(mensaje):
    tamano = len(mensaje)
    clave = ""

    for i in range(tamano):
        clave += random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

    return clave

In [63]:
# Vigenere de la tarea del 20%

def normalizar_texto(msj):
    return ''.join(c.upper() for c in msj if c.isalpha())

# ciframos con vigenere el texto a partir de la clave 
def cifrar_vigenere(msj, k):
    msj = normalizar_texto(msj)
    k = normalizar_texto(k)
    resultado = ""
    
    for i, msji in enumerate(msj):
        # Añadimos esta comprobacion para evitar que se introduccan caracteres no alfabeticos -> . , etc
        if msji.isalpha():
            # Obtenemos el valor de la posición i de la clave
            ki = k[i % len(k)] # Ki(mod(m)
            
            # Convertimos el char del texto y de la clave a su correspondiente valor ascii y restamos
            # el valor de la primera posición(A) para tener los valores entre 0 y 25
            msj_num = ord(msji) - ord('A')
            k_num = ord(ki) - ord('A')
            
            # Aplicamos el cifrado
            ci_num = (msj_num + k_num) % 26
            
            # Convertimos el número cifrado a char de nuevo y lo añadimos al resultado
            resultado += chr(ci_num + ord('A'))
    
    return resultado

# desciframos con vigenere el texto a partir de la clave
def descifrar_vigenere(c, k):
    c = normalizar_texto(c)
    k = normalizar_texto(k)
    resultado = ""
    
    for i, ci in enumerate(c):
        if ci.isalpha():
            # Obtenemos el valor de la posición i de la clave
            ki = k[i % len(k)]
            
            # Convertimos el char del texto y de la clave a su correspondiente valor ascii y restamos
            # el valor de la primera posición(A) para tener los valores entre 0 y 25            
            ci_num = ord(ci) - ord('A')
            ki_num = ord(ki) - ord('A')
            
            # Aplicamos descifrado
            msj_i_num = (ci_num - ki_num) % 26
            
            # Convertimos el número cifrado a char de nuevo y lo añadimos al resultado
            resultado += chr(msj_i_num + ord('A'))
        else:
            resultado += ci
    
    return resultado

**PROBAMOS LAS FUNCIONES DE GENERACIÓN DE CLAVE**

In [64]:
# Probando las funciones

cadena1 = "Prueba de cifrado Vigenere 1717 - > - ! "
clave1 = generaClaveVigenereConNoAlfabeticos(cadena1)
clave2 = generaClaveVigenereDefault(cadena1)

cifrado1 = cifrar_vigenere(cadena1, clave1)
descifrado1 = descifrar_vigenere(cifrado1, clave1)

cifrado2 = cifrar_vigenere(cadena1, clave2)
descifrado2 = descifrar_vigenere(cifrado2, clave2)

print("Cadena original: ", cadena1)
print("\n")
print("Clave con no alfabeticos: ", clave1)
print("Cifrado con clave con no alfabeticos: ", cifrado1)
print("Descifrado con clave con no alfabeticos: ", descifrado1)
print("\n")
print("Clave default: ", clave2)
print("Cifrado con clave default: ", cifrado2)
print("Descifrado con clave default: ", descifrado2)

Cadena original:  Prueba de cifrado Vigenere 1717 - > - ! 


Clave con no alfabeticos:  CLJXEQ WM ZHTPBPM IVQZXDSQ 1717 - > - ! 
Cifrado con clave con no alfabeticos:  RCDBFQZQBPYGBSADDWDKHJU
Descifrado con clave con no alfabeticos:  PRUEBADECIFRADOVIGENERE


Clave default:  PVJWMZRNWULAFDFJLHFIDPAGNHMEOLUZYRUCIWNH
Cifrado con clave default:  EMDANZURYCQRFGTETNJVHGE
Descifrado con clave default:  PRUEBADECIFRADOVIGENERE


## b.1 
**Idee y describa un medio de transmitir eficazmente esa clave a un copartícipe evitando el ataque del tercero interpuesto.**

Para poder transmitir la clave evitando que un tercero se haga con ella vamos a recurrir al protocolo de intercambio de claves secretas Diffie-Hellman.

Para esto se elegirán un primo grande(n) y una base(g), para los que se deberá cumplir 1<g<n.
Tras esto los usuarios elegirán sus claves privadas, el A elegirá el número secreto (x) y el usuario B elegirá el número secreto(y), los cuáles utilizarán para generar sus claves públicas que posteriormente intercambiarán.

El usuario A enviará X=g^x(mod n) al usario B y el usuario B enviará Y=g^y(mod n) al usario A.

Estas claves se usarán para generar una clave común entre ambos. El usuario A generará la clave común k=Y^x(mod n) y el usuario B k'=X^y(mod n).
Además como estas claves son comúnes deben cumplir:
k=k'=g^(xy) (mod n)

## a.2
**Mediante una función de argumento la longitud del texto a cifrar implementada por usted con la técnica que diseñó en el bloque anterior de esta pregunta, genere una clave tan larga como el mensaje plano que nos ocupa; seguidamente cífrelo con ella según Vigenère.**

In [65]:
mensaje_ejemplo = "SENORDIJOELCAPITANNEMOMOSTRANDOMELOSINSTRUMENTOSCOLGADOSDELASPAREDESDESUCAMAROTEHEAQUILOSAPARATOSEXIGIDOSPORLANAVEGACIONDELNAUTILUSALIGUALQUEENELSALONLOSTENGOAQUIBAJOMISOJOSINDICANDOMEMISITUACIONYMIDIRECCIONEXACTASENMEDIODELOCEANOALGUNOSDEELLOSLESONCONOCIDOSCOMOELTERMOMETROQUEMARCALATEMPERATURAINTERIORDELNAUTILUSELBAROMETROQUEPESAELAIREYPREDICELOSCAMBIOSDETIEMPOELHIGROMETROQUEREGISTRAELGRADODESEQUEDADDELAATMOSFERAELSTORMGLASSCUYAMEZCLAALDESCOMPONERSEANUNCIALAINMINENCIADELASTEMPESTADESLABRUJULAQUEDIRIGEMIRUTAELSEXTANTEQUEPORLAALTURADELSOLMEINDICAMILATITUDLOSCRONOMETROSQUEMEPERMITENCALCULARMILONGITUDYPORULTIMOMISANTEOJOSDEDIAYDENOCHEQUEMESIRVENPARAESCRUTARTODOSLOSPUNTOSDELHORIZONTECUANDOELNAUTILUSEMERGEALASUPERFICIEDELASAGUASSONLOSINSTRUMENTOSHABITUALESDELNAVEGANTEYSUUSOMEESCONOCIDOREPUSEPEROHAYOTROSAQUIQUERESPONDENSINDUDAALASPARTICULARESEXIGENCIASDELNAUTILUSESECUADRANTEQUEVEORECORRIDOPORUNAAGUJAINMOVILNOESUNMANOMETROESUNMANOMETROENEFECTOPUESTOENCOMUNICACIONCONELAGUACUYAPRESIONEXTERIORINDICADATAMBIENLAPROFUNDIDADALAQUESEMANTIENEMIAPARATO"

In [66]:
clave_ejemplo = generaClaveVigenereDefault(mensaje_ejemplo)

cifrado_ejemplo = cifrar_vigenere(mensaje_ejemplo, clave_ejemplo)
print("Mensaje ejemplo: ", mensaje_ejemplo)
print("Clave ejemplo: ", clave_ejemplo)
print("Cifrado ejemplo: ", cifrado_ejemplo)

descifrado_ejemplo = descifrar_vigenere(cifrado_ejemplo, clave_ejemplo)
print("Descifrado ejemplo: ", descifrado_ejemplo)

Mensaje ejemplo:  SENORDIJOELCAPITANNEMOMOSTRANDOMELOSINSTRUMENTOSCOLGADOSDELASPAREDESDESUCAMAROTEHEAQUILOSAPARATOSEXIGIDOSPORLANAVEGACIONDELNAUTILUSALIGUALQUEENELSALONLOSTENGOAQUIBAJOMISOJOSINDICANDOMEMISITUACIONYMIDIRECCIONEXACTASENMEDIODELOCEANOALGUNOSDEELLOSLESONCONOCIDOSCOMOELTERMOMETROQUEMARCALATEMPERATURAINTERIORDELNAUTILUSELBAROMETROQUEPESAELAIREYPREDICELOSCAMBIOSDETIEMPOELHIGROMETROQUEREGISTRAELGRADODESEQUEDADDELAATMOSFERAELSTORMGLASSCUYAMEZCLAALDESCOMPONERSEANUNCIALAINMINENCIADELASTEMPESTADESLABRUJULAQUEDIRIGEMIRUTAELSEXTANTEQUEPORLAALTURADELSOLMEINDICAMILATITUDLOSCRONOMETROSQUEMEPERMITENCALCULARMILONGITUDYPORULTIMOMISANTEOJOSDEDIAYDENOCHEQUEMESIRVENPARAESCRUTARTODOSLOSPUNTOSDELHORIZONTECUANDOELNAUTILUSEMERGEALASUPERFICIEDELASAGUASSONLOSINSTRUMENTOSHABITUALESDELNAVEGANTEYSUUSOMEESCONOCIDOREPUSEPEROHAYOTROSAQUIQUERESPONDENSINDUDAALASPARTICULARESEXIGENCIASDELNAUTILUSESECUADRANTEQUEVEORECORRIDOPORUNAAGUJAINMOVILNOESUNMANOMETROESUNMANOMETROENEFECTOPUESTOENCOMUNICACIONCONELAGUACUY

## b.2

**Atáquelo con las herramientas del laboratorio de criptoanálisis que implementó en la mencionada tarea.**

Primeramente, procedemos a traernos todo el código desarrollado durante la tarea del 20%. Este código ha sido adaptado en lo que a rutas respecta pero por lo demás se mantiene intacto.

In [67]:
# Funciones auxiliares: TRAIDAS DEL ARCHIVO DE LA TAREA DEL 20%
 
# Funciones auxiliares para análisis criptográfico del cifrado de Vigenère.
# Incluye herramientas estadísticas basadas en frecuencias de letras en español,
# cálculo del índice de coincidencia (IC), test chi-cuadrado, y utilidades
# matemáticas para el criptoanálisis. Estas funciones son la base para los
# métodos de Kasiski e Índice de Coincidencia.

import math

# Frecuencias esperadas de letras en español (%)
FRECUENCIAS_ESPANOL = {
    'E': 13.68, 'A': 12.53, 'O': 8.68, 'S': 7.98, 'R': 6.87,
    'N': 6.71, 'I': 6.25, 'D': 5.86, 'L': 4.97, 'C': 4.68,
    'T': 4.63, 'U': 3.93, 'M': 3.15, 'P': 2.51, 'B': 1.42,
    'G': 1.01, 'V': 0.90, 'Y': 0.90, 'Q': 0.88, 'H': 0.70,
    'F': 0.69, 'Z': 0.52, 'J': 0.44, 'X': 0.22, 'W': 0.02,
    'K': 0.01
}

# Índice de coincidencia esperado en español
IC_ESPANOL = 0.0775
IC_ALEATORIO = 0.0385

# Convierte texto a mayúsculas y elimina caracteres no alfabéticos y espacios
# def normalizar_texto(msj):
#     return ''.join(c.upper() for c in msj if c.isalpha())

# Cuenta la frecuencia de cada elemento en una lista
def contar_elementos(lista):
    recuento = {}
    for elemento in lista:
        recuento[elemento] = recuento.get(elemento, 0) + 1
    return recuento

# Calcula frecuencias absolutas de cada letra en el texto
def calcular_frecuencias(msj):
    msj = normalizar_texto(msj)
    return contar_elementos(msj)

# Calcula frecuencias relativas (porcentaje) de cada letra
def calcular_frecuencias_relativas(msj):
    frecuencias = calcular_frecuencias(msj)
    total = sum(frecuencias.values())
    
    if total == 0:
        return {}
    
    frecuencias_relativas = {}
    for letra, count in frecuencias.items():
        frecuencias_relativas[letra] = (count / total) * 100

    return frecuencias_relativas


# Calcula el índice de coincidencia (IC) de un texto
# Fórmula: IC = Σ(ni * (ni-1)) / (N * (N-1))
# Mide la probabilidad de que dos letras elegidas al azar sean iguales
def indice_coincidencia(msj):
    msj = normalizar_texto(msj)
    N = len(msj)
    
    if N <= 1:
        return 0.0
    
    frecuencias = contar_elementos(msj)
    suma = 0
    for f in frecuencias.values():
        suma += f * (f - 1)
    
    denominador = N * (N - 1)
    return suma / denominador


# Calcula el índice de coincidencia mutua (ICM) entre dos textos
# Fórmula: ICM = Σ(ni * mi) / (N * M)
# Mide la correlación de frecuencias entre dos textos
def indice_coincidencia_mutua(msj1, msj2):
    msj1 = normalizar_texto(msj1)
    msj2 = normalizar_texto(msj2)
    
    N = len(msj1)
    M = len(msj2)
    
    if N == 0 or M == 0:
        return 0.0
    
    freq1 = contar_elementos(msj1)
    freq2 = contar_elementos(msj2)
    
    suma = 0
    for letra in set(freq1) | set(freq2):
        suma += freq1.get(letra, 0) * freq2.get(letra, 0)
    
    return suma / (N * M)


# Calcula el test chi-cuadrado para comparar frecuencias observadas con las esperadas en español
# Fórmula: χ² = Σ((observado - esperado)² / esperado)
# Un valor bajo indica que el texto tiene distribución de frecuencias similar al español
def chi_cuadrado(msj):
    freq_observadas = calcular_frecuencias_relativas(msj)
    
    chi2 = 0.0
    for letra in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
        observada = freq_observadas.get(letra, 0)
        esperada = FRECUENCIAS_ESPANOL.get(letra, 0.01)
        
        if esperada > 0:
            chi2 += ((observada - esperada) ** 2) / esperada
    
    return chi2


# Divide el texto en grupos según la longitud de clave
# Cada grupo contiene las letras cifradas con la misma posición de clave
def dividir_en_grupos(msj, longitud_clave):
    msj = normalizar_texto(msj)
    grupos = []
    for i in range(longitud_clave):
        grupos.append('')
    
    for i, char in enumerate(msj):
        grupos[i % longitud_clave] += char
    
    return grupos


# Calcula el máximo común divisor usando el algoritmo de Euclides
def calcular_mcd(n1, n2):
    while n2:
        n1, n2 = n2, n1 % n2
    return n1

# Calcula el mínimo común múltiplo
def calcular_mcm(n1, n2):
    return abs(n1 * n2) // calcular_mcd(n1, n2)

# Encuentra todos los factores de un número hasta un máximo
def factores(n, max_factor=None):
    if max_factor is None:
        max_factor = n
    
    factores_lista = []
    for i in range(1, min(int(math.sqrt(n)) + 1, max_factor + 1)):
        if n % i == 0:
            factores_lista.append(i)
            if i != n // i and n // i <= max_factor:
                factores_lista.append(n // i)
    
    return sorted(factores_lista)

# Muestra una tabla de frecuencias del texto
def imprimir_tabla_frecuencias(msj):
    freq = calcular_frecuencias_relativas(msj)
    
    print("\n--- Frecuencias del texto ---")
    for letra in sorted(freq.keys()):
        print(f"{letra}: {freq[letra]:>6.2f}%")
    print()

# Formatea texto en líneas de ancho especificado
def formatear_texto(msj, ancho=60):
    lineas = []
    for i in range(0, len(msj), ancho):
        lineas.append(msj[i:i+ancho])
    return '\n'.join(lineas)

# Muestra el mensaje descifrado con su clave
def imprimir_mensaje_descifrado(mensaje, k):
    print("\n--- Mensaje descifrado ---")
    print(f"Clave: {k}")
    print(formatear_texto(mensaje, 66))
    print()

# Convierte un número de desplazamiento (0-25) a su letra correspondiente (A-Z)
def desplazamiento_a_letra(desplazamiento):
    return chr((desplazamiento % 26) + ord('A'))

# Convierte una letra (A-Z) a su desplazamiento numérico (0-25)
def letra_a_desplazamiento(letra):
    return ord(letra.upper()) - ord('A')

In [68]:

# INDICE DE COINCIDENCIA : TRAIDO DEL ARCHIVO DE LA TAREA DEL 20%

# Estima la longitud de la clave usando el método del Índice de Coincidencia
def estimar_longitud_clave_ic(c, max_longitud=20):
    # aceptamos solo letras, las cinvertimos en mayusculas y limpiamos espacios
    msj = normalizar_texto(c)
    resultados = []
    
    # iteramos sobre las posibles longitudes
    for longitud in range(1, max_longitud + 1):
        # dividimos en grupos de longitud estimada
        grupos = dividir_en_grupos(msj, longitud)
        
        ics = []
        for grupo in grupos:
            if len(grupo) > 1:
                ic = indice_coincidencia(grupo)
                ics.append(ic)
        
        if not ics:
            continue
        
        ic_promedio = sum(ics) / len(ics)
        diferencia = abs(ic_promedio - IC_ESPANOL)
        
        resultados.append({
            'longitud': longitud,
            'diferencia': diferencia
        })
        
    resultados.sort(key=lambda x: x['diferencia'])
    
    return resultados[0]['longitud']

# Encuentra la clave usando el Índice de Coincidencia Mutua (MIC)
# Teoría: Se basa en la Sección 2.2 del Segundo Libro. En lugar de atacar cada columna 
# por separado contra el español (lo que falla en textos cortos), alineamos las columnas entre sí.
def encontrar_clave_ic_mutua(c, longitud_clave):
    msj = normalizar_texto(c)
    
    # Teoría: Dividimos el texto en 'm' bloques.
    # Si la longitud es correcta, cada 'grupo' es un texto cifrado con un Cifrado César simple (monoalfabético).
    grupos = dividir_en_grupos(msj, longitud_clave)
    
    # FASE 1: DESPLAZAMIENTOS RELATIVOS
    # Objetivo: No buscamos aún la letra real (A, B...), sino la DISTANCIA entre la clave de la columna 0
    # y las claves de las demás columnas. Queremos que todas las columnas "giren" en sincronía.
    desplazamientos = [0] * longitud_clave
    
    # Tomamos la Columna 0 como "Ancla" o referencia fija.
    for i in range(1, longitud_clave):
        mejor_shift = 0
        mejor_icm = 0.0
        
        # Probamos los 26 desplazamientos relativos posibles (sigma en la teoría)
        for shift in range(26):            
            grupo_i_descifrado = ""
            sub_criptograma_actual = grupos[i]
            
            # Aplicamos el desplazamiento temporal a la columna i
            for j in sub_criptograma_actual:
                c_valor = ord(j) - ord('A')
                p_valor = c_valor - shift # Desplazamos hacia atrás
                descifrado_num = (p_valor % 26 + 26) % 26
                descifrado_char = chr(descifrado_num + ord('A'))
                grupo_i_descifrado += descifrado_char
            
            # Teoría: Calculamos el Índice de Coincidencia Mutuo (MIC) entre la Columna 0 y la Columna i desplazada.
            # Si el shift es correcto, ambas columnas tendrán la misma distribución de frecuencias subyacente.
            # Un MIC alto (~0.065) indica que hemos "alineado" las dos columnas estadísticamente.
            icm = indice_coincidencia_mutua(grupos[0], grupo_i_descifrado)
            
            if icm > mejor_icm:
                mejor_icm = icm
                mejor_shift = shift
        
        # Guardamos el desplazamiento relativo: "La columna i está rotada 'mejor_shift' posiciones respecto a la columna 0"
        desplazamientos[i] = mejor_shift
        
    
    # FASE 2: DESPLAZAMIENTO ABSOLUTO
    # Objetivo: Ahora que todas las columnas están alineadas relativamente con la Columna 0,
    # solo necesitamos romper el cifrado César de la Columna 0 para resolver todo el sistema.
    mejor_shift_absoluto = 0
    mejor_ic = 0.0
    
    for shift in range(26):
        grupo_0_descifrado = ""
        sub_criptograma_actual = grupos[0]
        
        # Desciframos la columna 0 con fuerza bruta (26 posibilidades)
        for j in sub_criptograma_actual:
            c_valor = ord(j) - ord('A')
            p_valor = c_valor - shift
            descifrado_num = (p_valor % 26 + 26) % 26
            descifrado_char = chr(descifrado_num + ord('A'))
            grupo_0_descifrado += descifrado_char
            
        # Teoría: Usamos el Índice de Coincidencia (IC) estándar (no mutuo).
        # Buscamos qué desplazamiento hace que la columna se parezca más al Español (IC ~ 0.074).
        ic = indice_coincidencia(grupo_0_descifrado)
        
        if ic > mejor_ic:
            mejor_ic = ic
            mejor_shift_absoluto = shift
    
    # FASE 3: RECONSTRUCCIÓN DE LA CLAVE
    # Teoría: Clave_i = (Clave_Base + Desplazamiento_Relativo_i) mod 26
    desplazamientos_finales = []

    for d in desplazamientos:  
        # Sumamos el desplazamiento base (absoluto) a los relativos que calculamos en la Fase 1
        desplazamiento_absoluto = (mejor_shift_absoluto + d) % 26
        desplazamientos_finales.append(desplazamiento_absoluto)

    clave = ""
    # Convertimos los números finales (0-25) a letras (A-Z)
    for d_num in desplazamientos_finales: 
        clave_char = chr(d_num + ord('A'))
        clave += clave_char
        
    return clave

# Encuentra la clave mediante análisis de frecuencias
def encontrar_clave_por_frecuencias(c, longitud_clave):
    msj = normalizar_texto(c)
    clave = []

    
    grupos = dividir_en_grupos(msj, longitud_clave)
    
    for pos in range(longitud_clave):
        grupo = grupos[pos]
        
        mejor_letra = 'A'
        mejor_puntuacion = float('inf')
        mejor_ic = 0.0
        
        for shift in range(26):
            descifrado = ""
            
            for j in grupo:
                j_valor = ord(j) - ord('A')
                
                p_valor = j_valor - shift
                
                descifrado_num = (p_valor % 26 + 26) % 26

                descifrado_char = chr(descifrado_num + ord('A'))
                
                descifrado += descifrado_char
                
            
            chi2 = chi_cuadrado(descifrado)
            ic = indice_coincidencia(descifrado)
            
            ic_normalizado = abs(ic - IC_ESPANOL)
            puntuacion = chi2 + ic_normalizado * 1000
            
            if puntuacion < mejor_puntuacion:
                mejor_puntuacion = puntuacion
                mejor_letra = chr(shift + ord('A'))
                mejor_ic = ic
        
        clave.append(mejor_letra)
    
    clave_str = ""

    for caracter in clave:
            clave_str += caracter    

    return clave_str

In [69]:
# MÉTODO DE KASISKI : TRAIDO DEL ARCHIVO DE LA TAREA DEL 20%

# Encuentra todas las secuencias repetidas en el texto cifrado
def encontrar_repeticiones(msj, min_long=3, max_long=5):
    
    msj = normalizar_texto(msj)
    
    repeticiones = {}

    # Buscar secuencias de diferentes longitudes
    for long_current in range(min_long, max_long + 1):
        for i in range(len(msj) - long_current + 1):
            seq = ""
            # Recorremos las subsecuencias de longitud long_current (en principio de 3 a 5)
            for j in range(long_current):
                seq += msj[i + j]
            # Añadimos la posicion en la que se encuentra la subsecuencia
            if seq in repeticiones:
                repeticiones[seq].append(i)
            else:
                repeticiones[seq] = [i]

    # Filtramos solo las que aparecen más de una vez
    repeticiones_filtradas = {}
    for seq, pos in repeticiones.items():
        if len(pos) > 1:
            # En repeticiones_filtradas, para una subsecuencia dada, guardamos la lista de posiciones donde aparece
            repeticiones_filtradas[seq] = pos

    return repeticiones_filtradas

# Calcula las distancias entre las posiciones de cada secuencia repetida    
def calcular_distancias(repeticiones):
    distancias = {}
    
    # Para cada secuencia repetida, calculamos las distancias entre sus posiciones
    for seq, posiciones in repeticiones.items():
        dists = []
        # Calculamos las distancias entre cada par de posiciones de una misma subsecuencia
        
        # Se hace en 2 bucles anidados porque queremos obtener las distancias de todas con todas
        for i in range(len(posiciones) - 1):
            for j in range(i + 1, len(posiciones)):
                dists.append(posiciones[j] - posiciones[i])
        distancias[seq] = dists
    
    return distancias

# Estima la longitud de la clave usando el método de Kasiski
def estimar_longitud_clave_kasiski(msj, min_long=3, max_long=7, max_longitud_clave=20):
    # Encontramos repeticiones
    repeticiones = encontrar_repeticiones(msj, min_long, max_long)
    
    # Calculamos distancias
    distancias_dict = calcular_distancias(repeticiones)
    
    # Juntamos todas las distancias en una sola lista
    todas_distancias = []
    for dists in distancias_dict.values():
        todas_distancias += dists
        
    # Contamos los factores de todas las distancias y luego recopilamos todos los factores de estos
    todos_factores = []
    for dist in todas_distancias:
        factores_distancias = factores(dist, max_longitud_clave)
        for f in factores_distancias:
            if f > 1:  # De nada serviría guardar el 1 como factor para todos
                todos_factores.append(f)
    
    # Contamos la frecuencia de los factores
    contador_factores = {}
    for factor in todos_factores:
        if factor in contador_factores:
            contador_factores[factor] += 1
        else:
            contador_factores[factor] = 1
    
    # Ordenar por frecuencia (ponemos la más común primero)
    factores_ordenados = sorted(contador_factores.items(), key=lambda x: x[1], reverse=True)
    
    # Devolvemos factores ordenados por frecuencia
    longitudes_probables = []
    for factor_ordenado in factores_ordenados:
        longitudes_probables.append(factor_ordenado[0])
    
    return longitudes_probables

# Encuentra la clave usando análisis de frecuencias para cada subgrupo
def encontrar_clave_por_frecuencias_kasiski(texto_cifrado, longitud_clave):
    texto = normalizar_texto(texto_cifrado)
    clave = []
    
    grupos = dividir_en_grupos(texto, longitud_clave)
    
    pos = 0
    while pos < longitud_clave:
        grupo = grupos[pos]
        mejor_letra = 'A'
        mejor_puntuacion = 999999.0
        
        shift = 0
        while shift < 26:
            descifrado = ''
            i = 0
            while i < len(grupo):
                c = grupo[i]
                nuevo_char = chr(((ord(c) - ord('A') - shift) % 26) + ord('A'))
                descifrado = descifrado + nuevo_char
                i = i + 1
            
            chi2 = chi_cuadrado(descifrado)
            
            if chi2 < mejor_puntuacion:
                mejor_puntuacion = chi2
                mejor_letra = chr(shift + ord('A'))
            
            shift = shift + 1
        
        clave.append(mejor_letra)
                
        pos = pos + 1
    
    clave_str = ''
    i = 0
    while i < len(clave):
        clave_str = clave_str + clave[i]
        i = i + 1
        
    return clave_str

# PROBAMOS A REVENTAR CON KASISKI

Vamos a tratar de averiguar la clave utilizada en nuestro cifrado_ejemplo usando Kasiski. Primero vamos a tratar de estimar la longitud y luego vamos a intentar recuperar la clave.

In [70]:
longitud_clave = estimar_longitud_clave_kasiski(cifrado_ejemplo, min_long=3, max_long=7, max_longitud_clave=20)
print(f"Longitud de clave estimada (Kasiski): {longitud_clave}")

# Probamos la mas probable:
clave_estimada_kasiski = encontrar_clave_por_frecuencias_kasiski(cifrado_ejemplo, longitud_clave[0])
print(f"Clave estimada para la frecuencia mas probable: {clave_estimada_kasiski}")
# Vemos que otras posibles claves estimadas habriamos obtenido:
for m in longitud_clave:
    clave_posible_kasiski = encontrar_clave_por_frecuencias_kasiski(cifrado_ejemplo, m)
    print(f"Longitud {m}: {clave_posible_kasiski}")




Longitud de clave estimada (Kasiski): [2, 3, 5, 6, 10, 4, 9, 7, 14, 8, 13, 20, 19, 11, 16, 17, 15, 12, 18]
Clave estimada para la frecuencia mas probable: RQ
Longitud 2: RQ
Longitud 3: ZEQ
Longitud 5: QQVET
Longitud 6: ZQOYEQ
Longitud 10: QQYGRKUMAT
Longitud 4: XKQG
Longitud 9: PQVNQQFUT
Longitud 7: IGZFECD
Longitud 14: YWALRQRCLQFTCG
Longitud 8: PQRGIKCZ
Longitud 13: FEXLQKGKFDMDV
Longitud 20: OLEEXKGULTQMYTRGXMYI
Longitud 19: BCEMKTRPYXEQQZOQTWH
Longitud 11: EEQYVTTEAZQ
Longitud 16: FXLENKBNLQDGTKPU
Longitud 17: ZZIPODOQVQFEPEVTV
Longitud 15: QUVXXQUDCMKYSET
Longitud 12: LQCYWKZUOQYQ
Longitud 18: IGVYYTVUOPMWCQQTLT


In [71]:
# Probamos a descifrar con las claves estimadas y comprobamos resultados
descifrado_kasiski = descifrar_vigenere(cifrado_ejemplo, clave_estimada_kasiski)
imprimir_mensaje_descifrado(descifrado_kasiski, clave_estimada_kasiski)


--- Mensaje descifrado ---
Clave: RQ
PGZTEMCDCGELQSZTDTTJAJKOPCXYDCOXWLFSIOMBSTTNISSXRMQIVLDBGXQTDVDPHJ
AFPOLJGBSXFDMIGLIUZAZUGBMQKKPHMNWWCMYJIDPHROKBHUMRIALUDGNLZCQYLOOI
JLXRWVJTPBACLUXRCGTPCYEZRQZQAAHCUAENHHQCTLOPSIGLEGFOXLHLRFBOTJFZVF
MXZNSGNNSWMGMQMOOUCTYFEOMHVOYGXSYFMSTWBNISXUMZLABPAXLNVJJVMULJODEW
ADQXMUTUCXFILMQQYEUHLPBHWXFINVLKPZPSSLLYWJDCZOAMBDPHUJPMEPSPLIYNVF
GYLRLLOZXQCCYOSFCVVPWUSMHPEEAJYDSGPOJELHGOFRCDDDZIVJCPTPBNBCPZYBPP
JHNMECVWODOIWGZAGKNXVCYDZWFMVYANKMUCGTIIIXZANHVZTFRINBWARUFSLTSWQU
PGZBBLDXXZKDPGKKHDOFGHMPOFGHQOKXLGTXYVAGMOPYPWFPHENQJSKLBEJQTDDVGD
DQOZQVBJFQNPGYTQDKUMYIIUQUHOOHWLPZHZVNCRNMQHICFHMFGXCASQEIGZJOTHST
GVHMEJCJZXASEHSFKDMXRIVVEBDRSSWASGMRTXXMNCNLAAIYIFLDJYYQMVJOXLGCNP
DHSNYGFEXBCJPIDQDOVNFUSPSHMRLHDPASHVSDADFTACDTNBRBOIZNNCEOQIFHPJUI
AFWLMYZCRJKHIRLWYEZHOLYWMRXQLHRYWWOPVSVUGKPYYFCSYGZJBSLIGIIZWIAJYQ
TSSGTDJZBNURUNTQHMRSVTASBPFPUBRDXGVFPWXWEBIZFFPOXUPGTSCZWOADJHIWLE
AFEJMHAXQIXGREFHADOENUEWFUICIXKLUUITLNPDUHZNLPZMXQUMLMBGAGJJAQGCQE
LZKHBUCMHBGRUTQXGJNRVYCR

In [72]:
longitud_clave_ic = estimar_longitud_clave_ic(cifrado_ejemplo)
print(f"Longitud de clave estimada (IC): {longitud_clave_ic}")

clave = encontrar_clave_por_frecuencias(cifrado_ejemplo, longitud_clave_ic)
print(f"Clave estimada: {clave}")  

clave_ic_mutua = encontrar_clave_ic_mutua(cifrado_ejemplo, longitud_clave_ic)
print(f"Clave estimada (IC mutua): {clave_ic_mutua}")

Longitud de clave estimada (IC): 19
Clave estimada: BCEMKTRPYXEQQZOQTWH
Clave estimada (IC mutua): AXHRRJNTRBTKCBBCPDN


In [73]:
# Probamos a descifrar con las claves estimadas y comprobamos resultados
descifrado_ic = descifrar_vigenere(cifrado_ejemplo, clave)
imprimir_mensaje_descifrado(descifrado_ic, clave)


--- Mensaje descifrado ---
Clave: BCEMKTRPYXEQQZOQTWH
FUMXLJCEVZRLRJCTBNDYPVPUNBZQXOPXONGPDXCPFXAKITLQEMRZYLBVQMFFIBBOJB
URQODLHYNGVRZMNIIVSTMUHSPQIEZWBZBCALABCPQHJQLYCDCFVESRDHGEMCRPOOMC
TAMDBBHSRTUOMUPTDDOYSMRDYNZRTTUCVRHNFBARIXTVQHIDYSGOPNIIMORCGNMWVG
FQMNTXQNQQWVBCRUMTELSRFOEJWLTPNGLJTPTXUGVSYLPZJULEPJQTTILNGGMJGFFT
VMGLZYARCYYBYMRHBESBVEQTBDDHPNFWQZHUTIGHMXQGGLANUWCHVASMCJCEAUDTTE
IQFDMLGBYNXLOCFJJSVQPNFMIGHEYDISHSUUHDNZAAGRUFEAURLXPTAMBOUVCZZSSP
HBXBTOACMCQAQSAAYMOUQLORMAMJVZTGXMVTJTGCSMOMSNTYVXLUOBOCSRABBHFAXR
PHSUOLEOAZIXZVZWMJMEIZGBPFYJRLFGBUGBFSAHFHCYQNIPFYXFYEPRZDLINPEVYF
ENJIGJONMNNQZRGQEBXMWCSJFGMUMGYDJLIZNPDOIVGVVGMEMGZQPATHHIETTDITXZ
EUJEYVDJRZBPZQITXHTURJOORBEIVSUUCVBDYDVLPUHXBAAAJCGMZMLUTSJPQETCOG
GHQHIVUQCHAIRAXCEONPGRNYIVZVSEDQTLUVTUDDDNKRSFSHPAQATZOCWQRFAQFXHM
HCWMFRMCSANHGLVLNQENMKAOGDYQDJSVRFEDIWCRGLIRLFDJBGXDLHAULOGYYAUVZQ
LUTDOMZNORBOUOMJUMSJYTYMLEUBZHPCZYPRQWPYFYDIVTCSERPHMLPZXFDDHBSLAQ
FLCIOZUJRIPISBAQQRBIUREXYNVCJONLSOSIAZUJSGBFFBAMPSVJGVRUNKQGARZVDE
MQNHZOM

In [74]:
descifrado_ic_mutua = descifrar_vigenere(cifrado_ejemplo, clave_ic_mutua)
imprimir_mensaje_descifrado(descifrado_ic_mutua, clave_ic_mutua)


--- Mensaje descifrado ---
Clave: AXHRRJNTRBTKCBBCPDN
GZJSETGACVCRFHPHFGXZUSKNXFVXTZVLMAUTWRDUCSTUMPSMPSFXLZFOKNKCDULSFI
QCWCBYVCGAWWWHGSMRZPXAVQCEMXTXGWWVKPWIYAWVHDZCVXDKSZLBHDNAXIFNBCQV
NBRAWURWNAQZSINGRHHSTROYRXDNAPFIJPUBJUUSNUOOALEKUDMCNAWMFISHDIFGZC
MMXTHVDBUJQWGZMNWXASOCLCCWKPMJOLIEMZXTBCGYMJCNNNFFUGLMDMHUCRSXESTX
OGHQWTTBGUFXJSFFOSWUPFVQWWNLLUBHWNFHHMZBNCNBZVEJBSNNJYFAGCWFFRYMDI
EXBOSZEOMRQFPHCECCZMWJQSWEUSCWCTMPPNRHJGWLMFSSSENLMCMOTWFKBRNFNQFD
LURCYLVVWGMHMDGOWZCYJFPWJVFTZVACISJRWHKVMNTJNGDCREHFUPMPGVTVCMCVQB
TDZQZRSMNNMQTWETHCWIEGCMVTWWFPYACZDWYCEDMDNEELVDJRRGDBKKJHHPJAKJWS
SRCCHOLIFXRMGNRWSZKAAVMKKDHNWKUKFWONLCRSBPHASBFOQCGMAGHFUWIMNENQSS
OYFLUGJXPMPTSKJYUCMEVFVKCHSGIGYNWWGATWFPLBDIHOYNXGZGARIPMCNLXAEICE
TVUACWZNXAKMNHTNKCLCUVGSJAWQLOHMAHFBHSQRHGESXCNAZEMHPKUQUDFJTKGCEH
AMAIMNXIGYAVKEPMSNZGWOWVCOEEBWGZKZFIFRVBKHPNWLRHOUBWFIFRGHQCUHQGFE
JHHHHGASLMUYYKTFFSGHLHCFFFZYUAZGVFLCWKNLTCWCWYZNXBTDTHAFLDQRLUMMFN
AEMMKGQUXWNVGFTKRWYDNBITFJGIXMAZWHMJFWPCCKXMBMGANFJNZPSZKFJQENGROK
AOAVDHG

## c.2
Valor y comente el resultado.

# CONCLUSIÓN:

Si el cifrado de Vigenère se realiza con una clave tan larga como el mensaje y generada de manera aleatoria, el cifrado se convierte en un cifrado de un solo uso que es teóricamente irrompible si se trata de una clave verdaderamente aleatoria, usada solo una vez y mantenida en secreto.

Dada esta cualidad teórica de seguridad, implementados los métodos que generan claves aleatorias de longitud igual al mensaje hemos podido comprobar que podemos cifrar y descifrar conociendo la clave y funciona correctamente. Sin embargo, al intentar romper el cifrado usando kasiski y indice de coincidencia, no hemos podido recuperar la clave ni descifrar el mensaje correctamente. Esto contrasta con los resultados obtenidos en la tarea del 20% donde si que pudimos recuperar la clave y descifrar el mensaje correctamente. Esto comprueba esta teoría de seguridad del cifrado de Vigenère con clave tan larga como el mensaje.

## Ejercicio 3

Sí, es posible implementar el protocolo al estilo Massey-Omura usando pkeyutl, esto es debido a su exponenciación en cuerpos finitos GF(q).

El primer lugar se elige un cuerpo finito de cardenal primo muy elevado GF(q) y cada usuario elegirá un número secreto aleatorio, 0 < e < q-1 tal que (e,q-1)=1.
Una vez elegidos cada usuario calculará d=e⁻¹ mod (q-1).

A envía a B el mensaje m cifrado como m^(eA) mod q.
openssl pkeyutl -derive -inkey alice_priv.pem -peerkey mensaje_m.pem -out c1.bin

B sobre-cifra el valor y devuelve a A s=m^(eAeB) mod q
openssl pkeyutl -derive -inkey bob_priv.pem -peerkey c1.bin -out s.bin

A elimina su clave al enviar r = s^(dA) mod q
openssl pkeyutl -derive -inkey alice_descifrado.pem -peerkey s.bin -out r.bin

B recibe r y calcula r^(dB) mod q
openssl pkeyutl -derive -inkey bob_descifrado.pem -peerkey r.bin -out mensaje_final.bin

No se recomienda este método porque el protocolo de Massey-Omura requiere que el receptor devuelva el mensaje cifrado al emisor, lo que genera una alta ineficiencia al necesitar tres intercambios de datos para un solo mensaje. Además, esta estructura lo hace extremadamente vulnerable a ataques Man-in-the-Middle, y el uso de pkeyutl resulta forzado ya que la herramienta está diseñada para estándares como el intercambio de claves de Diffie y Hellman y no para la gestión manual de exponentes inversos.

## Ejercicio 4

En una sesión SSH, lo primero que ocurre es el establecimiento de un canal seguro mediante la negociación de los algoritmos. Cliente y servidor intercambian sus versiones del protocolo y las listas de algoritmos que soportan para el intercambio de claves. A partir de estos intercambios se seleccionan algoritmos comunes y se ejecuta un mecanismo de intercambio de claves, normalmente basado en Diffie-Hellman pero más modernamente implementa variantes elípticas, que permiten a ambas partes calcular un secreto compartido sin transmitirlo directamente por la red. De este secreto se derivan las claves de sesión que se usarán para cifrar y autenticar toda la comunicación posterior.

En el lado del servidor, este demuestra su identidad firmando parte del intercambio de claves con su clave privada, y el cliente verifica dicha firma usando la clave pública correspondiente, que puede consultarse en el archivo known_hosts. Este paso es el que garantiza la autenticación, ya que asegura al cliente que está estableciendo la conexión con el servidor legítimo y no con un tercero malicioso. Una vez completada esta verificación, el servidor se ha autenticado con el cliente.

Solo después de que el canal seguro esté completamente establecido se lleva a cabo la autenticación del cliente. El protocolo SSH soporta varios métodos. El más seguro es la clave pública, donde el servidor verifica una firma generada con la clave privada del cliente. Por otra parte, en otro tipo de intercambios, se puede realizar mediante contraseña, donde el servidor valida el hash de las credenciales enviadas a través del túnel cifrado. Si el método elegido tiene éxito, el servidor concede el acceso y se inicia la interacción.


## Ejercicio 5

Es computacionalmente mucho más fácil encontrar una colisión que invertir la función para una salida específica.
Para entender por qué, comparamos el esfuerzo computacional (fuerza bruta) necesario en un escenario de seguridad ideal para una función hash con una salida de n bits (donde el espacio total de salidas es N=2n).

1.Invertir la función (Resistencia a la pre-imagen):
El objetivo es, dado un valor hash w específico, encontrar exactamente un x tal que H(x)=w. Como H se comporta (idealmente) como una función aleatoria, para tener una probabilidad del 50% de encontrar ese x específico, debes probar aproximadamente la mitad del espacio de claves. Esto tendría una complejidad de O(2n).

2.Encontrar una colisión:
El objetivo es, encontrar cualquier par x1​,x2​ tal que H(x1​)=H(x2​). No nos importa cuál sea el valor hash resultante, solo que coincidan. Debido a la Paradoja del Cumpleaños (que explicaremos abajo), la probabilidad de coincidencia aumenta mucho más rápido que la probabilidad de encontrar un valor fijo. Esto tendría una complejidad de O(2n/2).

Esto quiere decir, que por ejemplo para una función hash de 256 bits (como SHA-256):

Para invertirla (romper su unidireccionalidad) necesitaríamos ≈2256 operaciones.

Para encontrar una colisión necesitaríamos ≈2128 operaciones. Dado que 2128 es menor que 2256, encontrar colisiones es el ataque más fácil.



El valor 1,2*2^(n/2) proviene del análisis matemático de la Paradoja del Cumpleaños.

Esta paradoja establece que en un grupo de personas, la probabilidad de que dos cumplan años el mismo día es sorprendentemente alta con un grupo pequeño. En criptografía, los días del año son los posibles valores hash (N=2n) y las personas son las entradas que probamos (k).

Derivación matemática:
Queremos encontrar el número de intentos k necesarios para que la probabilidad de encontrar una colisión sea del 50% (p≈0.5). La probabilidad de que no haya colisión después de elegir k valores aleatorios de un universo de N posibilidades se aproxima mediante:

P(no colision)≈e^(-k²/2N)​

Por tanto, la probabilidad de que sí haya colisión es: 

p≈1−e^(-k²/2N)​

Si fijamos nuestra probabilidad objetivo en el 50% (0,5), la ecuación queda así: 

0,5=1−e^(-k²/2N)​

Despejamos para encontrar k (el número de intentos): 

e^(-k²/2N)​=0,5

Tomamos el logaritmo natural (ln) en ambos lados:

(-k²/2N)​=ln(0,5)=−ln(2)

k²=2N⋅ln(2)

k=(2N⋅ln(2))^(½)​

Aquí está la clave. Sabemos que ln(2)≈0,693. Sustituimos:

k≈(2⋅0,693⋅N​)^(½)​

k≈(1,386⋅N​)^(½)​

k≈1,177⋅(N)^(½)​​

Dado que N=2^n, entonces (N)^(½)​​=(2^n)^(½)=2^(n/2).

Redondeando 1,177 a 1,2, obtenemos la fórmula final de la seguridad ideal ante colisiones:

k≈1,2*2^(n/2)

